# Part 2: Matching Pictures and Words with CLIP

**Today we learn** how a model can match a photo to words it was never trained on. **We do** three matching games, break the model on purpose, then share a picture as a vector in a class game.

**Example.** Show CLIP a photo of a flower and five names: sunflower, rose, tulip, daisy, lavender. It scores each name and picks the closest. Nobody trained it on your photo.

You used cosine similarity in Part 1. Same idea, bigger vectors: 512 numbers instead of 14.

Time: about 60 minutes. No GPU needed.

In [ ]:
!pip install -q torch transformers scikit-learn scikit-image matplotlib pillow

## Setup

**Why one shared setup?** A comparison only means something when both vectors come from the same model. CLIP has an image reader and a text reader that end in the same 512-number space, so every game below uses just these two functions.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics.pairwise import cosine_similarity


NAME = "openai/clip-vit-base-patch32"
clip, proc = CLIPModel.from_pretrained(NAME), CLIPProcessor.from_pretrained(NAME)

def load(x):                                   # accepts a file path or a PIL image
    return x if isinstance(x, Image.Image) else Image.open(x).convert("RGB")

def _vec(o):                                   # some library versions wrap the result in an object
    return o if torch.is_tensor(o) else o.pooler_output

def embed_images(items):                       # list of paths or images, returns (n, 512)
    with torch.no_grad():
        return _vec(clip.get_image_features(**proc(images=[load(i) for i in items], return_tensors="pt"))).numpy()

def embed_texts(texts):                        # list of strings, returns (n, 512)
    with torch.no_grad():
        return _vec(clip.get_text_features(**proc(text=texts, return_tensors="pt", padding=True))).numpy()

## Get some pictures

**Why starter images?** So every group can run every game in the first minute. Swap in your own photos afterwards.

The dog comes from the PyTorch Hub examples. The others ship inside the `scikit-image` and `scikit-learn` libraries.

In [ ]:
import os
from PIL import ImageOps
from skimage import data
from sklearn.datasets import load_sample_image

os.makedirs("starter", exist_ok=True)
!wget -q -O starter/dog.jpg https://github.com/pytorch/hub/raw/master/images/dog.jpg
pics = {"cat": data.chelsea(), "coffee": data.coffee(), "astronaut": data.astronaut(),
        "rocket": data.rocket(), "flower": load_sample_image("flower.jpg"), "temple": load_sample_image("china.jpg")}
for name, arr in pics.items():
    Image.fromarray(arr).save(f"starter/{name}.jpg")
ImageOps.mirror(load("starter/dog.jpg")).save("starter/dog_mirrored.jpg")

names = sorted(f[:-4] for f in os.listdir("starter"))
fig, axes = plt.subplots(1, len(names), figsize=(2.2 * len(names), 2.4))
for ax, nm in zip(axes, names):
    ax.imshow(load(f"starter/{nm}.jpg")); ax.set_title(nm, fontsize=8); ax.axis("off")
plt.show()

## Game 1: Name the plant from one image

*Context: you found a plant and want its name.*

**Why a sentence like "a photo of a ..."?** CLIP learned from photos paired with captions, so text that looks like a caption matches better than a bare word.

Fill in the scoring and the winner.

In [ ]:
img = embed_images(["starter/flower.jpg"])
plants = ["sunflower", "rose", "tulip", "daisy", "lavender"]
txt = embed_texts([f"a photo of a {p}" for p in plants])

# TODO: scores = cosine similarity between img and txt, a row of 5 numbers
# TODO: print every plant with its score, then print the best one (np.argmax)

<details><summary>Solution</summary>

```python
scores = cosine_similarity(img, txt)[0]
for p, s in zip(plants, scores): print(p, round(float(s), 3))
print("Best:", plants[np.argmax(scores)])
```

</details>

Your turn: photograph a plant near you, upload it in the Files panel on the left, and change the path.

## Game 2: Find the tool in the inventory

*Context: you need the right piece from a box of parts.*

**Why embed every image once?** Embedding is the slow step. After that, a search is just a cheap comparison, which is how photo search works in real apps.

The starter pictures stand in for an inventory. Then photograph 6 to 8 real tools into a folder `inventory/` and run it again.

In [ ]:
import glob
paths = sorted(glob.glob("starter/*.jpg"))
image_vectors = embed_images(paths)                 # embed once
query = embed_texts(["a photo of a cup of coffee"])

# TODO: score the query against every image vector, then print the paths from best to worst
#   hint: np.argsort(scores)[::-1]

<details><summary>Solution</summary>

```python
scores = cosine_similarity(query, image_vectors)[0]
for i in np.argsort(scores)[::-1]: print(paths[i], round(float(scores[i]), 3))
```

</details>

Try "a photo of an astronaut", then "something to drink from". Which phrasing works?

## Game 3: Pick the right part from a reference image

*Context: you hold one part and must find the same one in a pile.*

**Why does image to image work?** Both pictures go through the same image reader, so two views of the same thing land close together. The mirrored dog is the same dog seen differently.

In [ ]:
ref = embed_images(["starter/dog.jpg"])
cands = [p for p in paths if "dog.jpg" not in p]        # everything except the reference itself

# TODO: compare ref with embed_images(cands) and print the best candidate with its score

<details><summary>Solution</summary>

```python
s = cosine_similarity(ref, embed_images(cands))[0]
print("Best:", cands[np.argmax(s)], round(float(s.max()), 3))
```

</details>

Notice that all three games used the same two steps: embed, then `cosine_similarity`. Only the inputs changed.

## Why it works: vectors and cosine similarity

**Why can pictures be compared with words?** During training, CLIP pushed each photo and its caption to the same spot on a map of meaning, over about 400 million pairs. Every image or text becomes 512 numbers, and cosine similarity measures how close two spots are.

**ELI5.** Two friends point at things. Same direction, they agree. At right angles, nothing in common. Opposite, they disagree. Arm length is ignored. This is the same demo from Part 1.

In [ ]:
from IPython.display import HTML

HTML("""<div style="font-family:system-ui,sans-serif;max-width:440px;margin:auto">
<label>Arrow A angle: <b id="cd_la">20</b>&deg;</label>
<input id="cd_sa" type="range" min="0" max="360" value="20" style="width:100%">
<label>Arrow B angle: <b id="cd_lb">70</b>&deg;</label>
<input id="cd_sb" type="range" min="0" max="360" value="70" style="width:100%">
<svg viewBox="-110 -110 220 220" width="220" height="220" style="display:block;margin:auto">
<circle r="90" fill="none" stroke="gray" stroke-opacity=".5"/>
<line id="cd_A" x1="0" y1="0" stroke="#d4561f" stroke-width="4" stroke-linecap="round"/>
<line id="cd_B" x1="0" y1="0" stroke="#2f5bea" stroke-width="4" stroke-linecap="round"/></svg>
<p style="text-align:center;margin:4px">cosine similarity = <b id="cd_cs" style="font-size:1.6em">0.64</b> &nbsp; <span id="cd_pct">64% match</span></p>
<p id="cd_msg" style="text-align:center;margin:4px;opacity:.8"></p></div>
<script>
(function(){var $=function(i){return document.getElementById(i)};
function u(){var a=+$("cd_sa").value,b=+$("cd_sb").value,r=Math.PI/180,c=Math.cos((a-b)*r);
$("cd_la").textContent=a;$("cd_lb").textContent=b;
$("cd_A").setAttribute("x2",90*Math.cos(a*r));$("cd_A").setAttribute("y2",-90*Math.sin(a*r));
$("cd_B").setAttribute("x2",90*Math.cos(b*r));$("cd_B").setAttribute("y2",-90*Math.sin(b*r));
$("cd_cs").textContent=c.toFixed(2);$("cd_pct").textContent=Math.round(c*100)+"% match";
$("cd_msg").textContent=c>=0.95?"Almost the same direction: twins!":c>0.5?"Pointing roughly the same way":c>-0.5?"Not much in common":"Pointing in opposite directions";}
$("cd_sa").oninput=$("cd_sb").oninput=u;u();})();
</script>""")

In [ ]:
v = embed_texts(["a photo of a dog"])
print("shape:", v.shape)
print("first 6 numbers:", v[0][:6].round(3))
print("length of the arrow:", round(float(np.linalg.norm(v)), 2))

## Take-home questions

**Why take-home?** Playing with a tool shows what it does. Answering these alone shows whether you understood it.

1. Your correct matches scored around 0.25 to 0.30, not 0.97. Why are CLIP scores so low? Does only the ranking matter?
2. Rewrite the Game 1 prompts as just the name, without "a photo of a". Does the winner or the gap change?
3. Add a sixth candidate that is not in the photo, like "a car". Can CLIP say "none of these"? How would you add that?
4. Find a pair of objects CLIP confuses. What do they have in common?
5. CLIP never saw your inventory. Why can it still work? (Hint: zero-shot.)
6. Draw the map of meaning in 2D with 8 of your items. Where would a text and its photo sit?

## Experiment A: Synonyms

**Why test synonyms?** If the model understands meaning, "puppy" and "hound" should behave like "dog". Seeing where that fails tells you how much it relies on exact words.

In [ ]:
dog = embed_images(["starter/dog.jpg"])
for t in ["a photo of a dog", "a photo of a puppy", "a photo of a hound",
          "a photo of a canine", "a photo of a pooch", "a photo of a cat"]:
    print(f"{t:28s}", round(float(cosine_similarity(dog, embed_texts([t]))[0][0]), 3))

Your turn: swap words in a sentence for synonyms, like "a small red car" to "a tiny crimson automobile". Which swaps barely matter, and which break it?

## Experiment B: Flip, rotate and grayscale

**Why transform the image?** A good model should not care if a dog is mirrored or the photo is black and white. Predict first which change moves the vector the most, then check.

In [ ]:
from PIL import ImageOps
img = load("starter/dog.jpg")
variants = {"original": img, "mirrored": ImageOps.mirror(img), "upside down": ImageOps.flip(img),
            "grayscale": ImageOps.grayscale(img).convert("RGB"), "rotated 90": img.rotate(90, expand=True)}
E = embed_images(list(variants.values()))
t = embed_texts(["a photo of a dog"])
for (k, _), e in zip(variants.items(), E):
    same = cosine_similarity([e], [E[0]])[0][0]
    word = cosine_similarity([e], t)[0][0]
    print(f"{k:12s} vs original {same:.3f} | vs 'dog' {word:.3f}")

## Experiment C: Break CLIP

**Why break it?** Every model has a boundary where it stops working. Finding that boundary, and recording exactly where it is, is how you learn when not to trust it.

Rotate and blur the dog until CLIP prefers "cat". Then try your own ideas: tiny crops, shrinking to 16 by 16 pixels and back, a sticky note reading "iPod" on an apple, counting ("three apples" vs "five apples"), left and right, or a negation like "a dog not wearing a hat".

In [ ]:
from PIL import ImageFilter
dog_img = load("starter/dog.jpg")
prompts = embed_texts(["a photo of a dog", "a photo of a cat"])

def dog_vs_cat(im):
    s = cosine_similarity(embed_images([im]), prompts)[0]
    return f"dog {s[0]:.3f} | cat {s[1]:.3f} | says {'dog' if s[0] > s[1] else 'cat'}"

for a in [0, 15, 45, 90, 135, 180]:
    print("rotate", a, dog_vs_cat(dog_img.rotate(a, expand=True)))
for r in [1, 3, 8, 15, 30]:
    print("blur  ", r, dog_vs_cat(dog_img.filter(ImageFilter.GaussianBlur(r))))

## Experiment D: Bias and mistakes

**Why look for bias?** CLIP learned from internet captions, so it inherits the stereotypes in them. If a model like this screened photos or CVs, those gaps could harm real people, so we measure them first. This probe needs no photos of people.

In [ ]:
jobs   = ["nurse", "engineer", "CEO", "teacher", "pilot", "cleaner", "scientist"]
groups = ["a man", "a woman"]
S = cosine_similarity(embed_texts([f"a photo of {g}" for g in groups]),
                      embed_texts([f"a photo of a {j}" for j in jobs]))
for j, m, w in zip(jobs, S[0], S[1]):
    print(f"{j:10s} man {m:.3f}  woman {w:.3f}  gap {m - w:+.3f}")

Your turn: try other pairs, such as age groups, nationalities, or "a rich person" and "a poor person" with objects or places. Where are the biggest gaps? Who could be harmed?

**Ethics rule:** use only photos you have consent for, and never label real people by sensitive traits. The goal is to study the model, not to judge people.

## Game: Guess My Image

**Why share vectors and not pictures?** A vector is about 1.4 KB of text, small enough for a chat message or a spreadsheet cell, and you choose what to reveal. A vector still leaks what the picture shows, so pick images you are happy to share and no photos of other people.

**How it works.** Everyone embeds one image and pastes a line `name,code` into a shared sheet. Nobody shows the picture. Then you find your twin by comparing vectors, and teams try to describe a mystery vector in words.

Everyone must use the same CLIP model above, or the scores mean nothing. The code stores numbers as 16-bit floats. Do not round to two decimals like in Part 1: unit-length CLIP values are around 0.04 and would be destroyed.

In [ ]:
import base64

def pack(v):                                     # vector to short text
    v = v / np.linalg.norm(v)                    # unit length, so cosine equals the dot product
    return base64.b64encode(v.astype(np.float16).tobytes()).decode()

def unpack(s):                                   # short text back to a vector
    return np.frombuffer(base64.b64decode(s), dtype=np.float16).astype(np.float32)

MY_NAME  = "your name"
MY_IMAGE = "starter/cat.jpg"                     # change to your own picture
my_code  = pack(embed_images([MY_IMAGE])[0])
print(f"{MY_NAME},{my_code}")                    # copy this line into the class sheet
print(len(my_code), "characters")

In [ ]:
# Paste every line from the class sheet between the triple quotes.
# While it is empty, three demo lines are used so the cells still run.
class_codes = """   """
if not class_codes.strip():
    class_codes = "\n".join(f"Demo {nm},{pack(embed_images([f'starter/{nm}.jpg'])[0])}" for nm in ["cat", "coffee", "rocket"])

people = {}
for line in class_codes.strip().splitlines():
    nm, code = line.rsplit(",", 1)
    people[nm.strip()] = unpack(code.strip())

mine = unpack(my_code)
ranking = sorted(((float(cosine_similarity([mine], [v])[0][0]), nm) for nm, v in people.items() if nm != MY_NAME), reverse=True)
for s, nm in ranking[:3]: print(f"{nm}: {s:.3f}")

**Round 2: describe the mystery image.** One person shares only a vector. Each team writes short prompts, and the code scores them against it. Highest score wins, then reveal the picture. Text scores are low, so only the ranking matters.

In [ ]:
def guess(prompt, target_name):
    return float(cosine_similarity(embed_texts([prompt]), [people[target_name]])[0][0])

print(guess("a photo of a cat on a lawn", "Demo cat"))
print(guess("a cup of coffee", "Demo cat"))

Debrief: can you describe a picture well enough for the model to recognise it? What did the model notice that you did not write? Could someone work out your image from your vector?

## Project idea: a visual search engine on a Kaggle dataset

**Why measure against labels?** A model that seems to work on three photos is a story. Accuracy on thousands of labelled photos is evidence, and it shows you exactly where the model fails.

**Dataset 1, easiest: Flowers Recognition.** About 4,200 photos in five folders: daisy, dandelion, rose, sunflower, tulip. The folder name is the correct answer. [kaggle.com/datasets/alxmamaev/flowers-recognition](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition)

**Dataset 2, bigger: Fashion Product Images Small.** About 44,000 product photos with a text name, category and colour for each. Good for text search. [kaggle.com/datasets/paramaggarwal/fashion-product-images-small](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small)

**Your project, in four steps**
1. Embed a sample of images, say 50 per class, and save the vectors with `np.save` so you never embed twice.
2. Classify every image with zero-shot prompts and compute accuracy against the folder labels.
3. Build a search box: type a sentence, show the top 5 images.
4. Find the classes CLIP confuses and test whether better prompts fix it. Report your findings in one page.

Downloading may ask you to log in to Kaggle.

In [ ]:
!pip install -q kagglehub
import kagglehub, glob, os

path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
files  = glob.glob(f"{path}/**/*.jpg", recursive=True)
labels = [os.path.basename(os.path.dirname(f)) for f in files]
print(len(files), "images, classes:", sorted(set(labels)))

## Appendix: definitions and reasons

| Term | What it is | Why we care |
|---|---|---|
| Vector | An ordered list of numbers | Lets a computer do geometry on anything |
| Embedding | A vector that places an item on a map of meaning | Similar meaning gives nearby numbers |
| Cosine similarity | The angle between two vectors, from 1 to -1 | Compares direction and ignores size |
| Zero-shot classification | Choosing among labels the model never trained on, by writing them as text | No labelled training data needed |
| Prompt template | A caption style sentence like "a photo of a rose" | Matches how CLIP was trained |
| float16 and base64 | A small number format, and a way to write bytes as text | Makes a vector short enough to paste |
| Bias | Systematic unfairness a model learns from its data | Matches can repeat stereotypes |